In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

## Ingestion del archivo "movie_genre.json"

###Paso 1 - Leer el archivo JSON usando "DataframeReader" de Spark

In [0]:
movie_genre_df = spark.read.schema("movieId INT, genreId INT").json(f"{bronze_folder_path}/{v_file_date}/movie_genre.json")

### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movie_genre_renamed_df = add_ingestion_date(movie_genre_df)\
    .withColumnsRenamed({"movieId": "movie_id", "genreId": "genre_id"})\
    .withColumn("environment", lit(v_environment))\
    .withColumn("file_date", lit(v_file_date))

### Paso 3 - Escribir la salida en un formato "Parquet" PartitionBy

In [0]:
#overwrite_partition("movie_silver", "movie_genres", "file_date", v_file_date)

In [0]:
#movie_genre_renamed_df.write.mode("overwrite").parquet(f"{silver_folder_path}/movie_genres")

In [0]:
#movie_genre_renamed_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movie_genres")

condition_merge = 'tgt.movie_id = src.movie_id AND tgt.genre_id = src.genre_id AND tgt.file_date = src.file_date'

incremental_merge("movie_silver", "movie_genres", movie_genre_renamed_df, condition_merge, "file_date")

In [0]:
%sql
SELECT file_date, count(1)
FROM movie_silver.movie_genres
GROUP BY file_date;

file_date,count(1)
2024-12-16,7000
2024-12-23,3000
2024-12-30,2160


In [0]:
dbutils.notebook.exit("Exitoso")